<a href="https://colab.research.google.com/github/vandanachegondi4-lang/IBN/blob/main/llama_trained.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [2]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Llama-3.1-8B-bnb-4bit",      # Llama-3.1 15 trillion tokens model 2x faster!
    "unsloth/Llama-3.1-8B-Instruct-bnb-4bit",
    "unsloth/Llama-3.1-70B-bnb-4bit",
    "unsloth/Llama-3.1-405B-bnb-4bit",    # We also uploaded 4bit for 405b!
    "unsloth/Mistral-Nemo-Base-2407-bnb-4bit", # New Mistral 12b 2x faster!
    "unsloth/Mistral-Nemo-Instruct-2407-bnb-4bit",
    "unsloth/mistral-7b-v0.3-bnb-4bit",        # Mistral v3 2x faster!
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/Phi-3.5-mini-instruct",           # Phi-3.5 2x faster!
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/gemma-2-9b-bnb-4bit",
    "unsloth/gemma-2-27b-bnb-4bit",            # Gemma 2x faster!
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.1-8B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.9: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.96G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/235 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/459 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

In [3]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2026.6.9 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [5]:
from datasets import load_dataset
dataset = load_dataset("json", data_files="llama_training.jsonl", split="train")
dataset = dataset.select(range(200))
print(dataset)

Dataset({
    features: ['instruction', 'input', 'output', 'response_format'],
    num_rows: 200
})


In [6]:
EOS_TOKEN = tokenizer.eos_token

prompt_template = """### Instruction:
{}

### Input:
{}

### Response:
{}"""

def formatting_prompts_func(examples):
    texts = []

    for instruction, inp, output in zip(
        examples["instruction"],
        examples["input"],
        examples["output"],
    ):
        texts.append(
            prompt_template.format(
                instruction,
                inp,
                output,
            ) + EOS_TOKEN
        )

    return {"text": texts}

dataset = dataset.map(
    formatting_prompts_func,
    batched=True,
)

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

In [7]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        output_dir="outputs",
        dataset_text_field="text",
        max_length=2048,

        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,

        learning_rate=2e-4,
        num_train_epochs=1,

        logging_steps=10,
        optim="adamw_8bit",

        report_to="none",
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/200 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [8]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.563 GB.
6.705 GB of memory reserved.


In [9]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 200 | Num Epochs = 1 | Total steps = 25
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Step,Training Loss
10,1.458000
20,0.516700


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!


In [10]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.563 GB.
6.705 GB of memory reserved.


In [51]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

149.2837 seconds used for training.
2.49 minutes used for training.
Peak reserved memory = 11.541 GB.
Peak reserved memory for training = 4.836 GB.
Peak reserved memory % of max memory = 79.249 %.
Peak reserved memory for training % of max memory = 33.207 %.


In [12]:
model.save_pretrained("intent_lora")
tokenizer.save_pretrained("intent_lora")

('intent_lora/tokenizer_config.json',
 'intent_lora/special_tokens_map.json',
 'intent_lora/tokenizer.json')

In [13]:
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 4096, padding_idx=128004)
        (layers): ModuleList(
          (0): LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear

In [31]:
user_input = input("Enter your intent: ")

prompt = f"""### Instruction:
Convert the following natural language intent into structured JSON with intent.domain, intent.operation, entities, parameters, and modifiers.

### Input:

{user_input}

### Response:
"""

Enter your intent: show hostname on device lrs40


In [32]:
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=256,
    temperature=0.1,
    do_sample=False,
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

### Instruction:
Convert the following natural language intent into structured JSON with intent.domain, intent.operation, entities, parameters, and modifiers.

### Input:

show hostname on device lrs40

### Response:
{
  "intent": {
    "domain": "system_mgmt",
    "operation": "read"
  },
  "entities": {
    "device": "lrs40"
  },
  "parameters": {},
  "modifiers": {
    "conditional": false
  }
}


In [56]:
import gc

gc.collect()
torch.cuda.empty_cache()

In [57]:
del model
del base_model
del trainer

In [58]:
gc.collect()
torch.cuda.empty_cache()

In [59]:
!nvidia-smi

Mon Jun 29 18:06:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   77C    P0             34W /   70W |   11927MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [60]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

149.2837 seconds used for training.
2.49 minutes used for training.
Peak reserved memory = 11.752 GB.
Peak reserved memory for training = 5.047 GB.
Peak reserved memory % of max memory = 80.698 %.
Peak reserved memory for training % of max memory = 34.656 %.


In [9]:
!ls

huggingface_tokenizers_cache  llama_training.jsonl  sample_data
intent_lora		      merged_model	    unsloth_compiled_cache
llama.cpp		      outputs


In [ ]:
import os
os.kill(os.getpid(), 9)

In [1]:
!ls

huggingface_tokenizers_cache  llama_training.jsonl  unsloth_compiled_cache
intent_lora		      outputs
llama.cpp		      sample_data


In [2]:
!rm -rf merged_model

In [2]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    "unsloth/Llama-3.1-8B",
    max_seq_length=2048,
    load_in_4bit=True,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.9: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [3]:
model.load_adapter("intent_lora")

In [5]:
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

prompt = """Convert the following natural language intent into structured JSON:
just to be clear, edit vlan id 567 on brs40-1"""

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

output = model.generate(**inputs, max_new_tokens=200)
print(tokenizer.decode(output[0]))

<|begin_of_text|>Convert the following natural language intent into structured JSON:
just to be clear, edit vlan id 567 on brs40-1

{
  "operation": "update",
  "device": "brs40-1",
  "vlan": 567
}<|end_of_text|>


In [6]:
model.save_pretrained_merged(
    "merged_model",
    tokenizer,
    save_method="merged_16bit"
)

Unsloth: Saving full fine-tuned model to 'merged_model' ...
Unsloth: Model saved successfully to 'merged_model'


In [7]:
!ls -lah merged_model

total 97M
drwxr-xr-x 2 root root 4.0K Jun 29 19:15 .
drwxr-xr-x 1 root root 4.0K Jun 29 19:15 ..
-rw-r--r-- 1 root root 1.3K Jun 29 19:15 adapter_config.json
-rw------- 1 root root  81M Jun 29 19:15 adapter_model.safetensors
-rw-r--r-- 1 root root  230 Jun 29 19:15 generation_config.json
-rw-r--r-- 1 root root  459 Jun 29 19:15 special_tokens_map.json
-rw-r--r-- 1 root root  50K Jun 29 19:15 tokenizer_config.json
-rw-r--r-- 1 root root  17M Jun 29 19:15 tokenizer.json


In [5]:
!pip install -q transformers sentencepiece protobuf

In [16]:
!git clone https://github.com/ggerganov/llama.cpp

Cloning into 'llama.cpp'...
remote: Enumerating objects: 101393, done.
remote: Counting objects: 100% (122/122), done.
remote: Compressing objects: 100% (93/93), done.
remote: Total 101393 (delta 71), reused 29 (delta 29), pack-reused 101271 (from 3)
Receiving objects: 100% (101393/101393), 403.55 MiB | 32.41 MiB/s, done.
Resolving deltas: 100% (71290/71290), done.


In [17]:
%cd llama.cpp

/content/llama.cpp/llama.cpp


In [19]:
!find /content -type d -name "merged_model"

/content/merged_model


In [20]:
!ls /content/merged_model

adapter_config.json	   generation_config.json   tokenizer_config.json
adapter_model.safetensors  special_tokens_map.json  tokenizer.json


In [21]:
!python convert_hf_to_gguf.py /content/merged_model --outtype f16 --outfile intent.gguf

INFO:hf-to-gguf:Loading model: merged_model
Traceback (most recent call last):
  File "/content/llama.cpp/llama.cpp/conversion/base.py", line 1051, in load_hparams
    config = AutoConfig.from_pretrained(dir_model, trust_remote_code=False).to_dict()
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/models/auto/configuration_auto.py", line 1329, in from_pretrained
    raise ValueError(
ValueError: Unrecognized model in /content/merged_model. Should have a `model_type` key in its config.json, or contain one of the following strings in its name: aimv2, aimv2_vision_model, albert, align, altclip, apertus, arcee, aria, aria_text, audio-spectrogram-transformer, autoformer, aya_vision, bamba, bark, bart, beit, bert, bert-generation, big_bird, bigbird_pegasus, biogpt, bit, bitnet, blenderbot, blenderbot-small, blip, blip-2, blip_2_qformer, bloom, bridgetower, bros, camembert, canine, chameleon, chinese_clip,

In [18]:
!python convert_hf_to_gguf.py ../merged_model --outtype f16 --outfile intent.gguf

ERROR:hf-to-gguf:Error: ../merged_model is not a directory


In [10]:
!mkdir -p intent-ollama
!mv intent.gguf intent-ollama/

mv: cannot stat 'intent.gguf': No such file or directory


In [12]:
!find /content -name "*.gguf"

/content/llama.cpp/models/ggml-vocab-gpt-neox.gguf
/content/llama.cpp/models/ggml-vocab-falcon.gguf
/content/llama.cpp/models/ggml-vocab-starcoder.gguf
/content/llama.cpp/models/ggml-vocab-baichuan.gguf
/content/llama.cpp/models/ggml-vocab-aquila.gguf
/content/llama.cpp/models/ggml-vocab-refact.gguf
/content/llama.cpp/models/ggml-vocab-gpt-2.gguf
/content/llama.cpp/models/ggml-vocab-deepseek-llm.gguf
/content/llama.cpp/models/ggml-vocab-deepseek-coder.gguf
/content/llama.cpp/models/ggml-vocab-llama-spm.gguf
/content/llama.cpp/models/ggml-vocab-mpt.gguf
/content/llama.cpp/models/ggml-vocab-command-r.gguf
/content/llama.cpp/models/ggml-vocab-gemma-4.gguf
/content/llama.cpp/models/ggml-vocab-bert-bge.gguf
/content/llama.cpp/models/ggml-vocab-qwen35.gguf
/content/llama.cpp/models/ggml-vocab-phi-3.gguf
/content/llama.cpp/models/ggml-vocab-llama-bpe.gguf
/content/llama.cpp/models/ggml-vocab-nomic-bert-moe.gguf
/content/llama.cpp/models/ggml-vocab-qwen2.gguf
